In [8]:
import os
import re
import sys

import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

ROOT_PATH = r'D:\Users\wangy\PycharmProjects\GlassDoor'

if ROOT_PATH not in sys.path:
    sys.path.append(ROOT_PATH)

from Constants import Constants as const

In [3]:
gd_monthly_df = pd.read_pickle(os.path.join(const.TEMP_PATH, 'glassdoor_monthly_compustat_ann_2008_2023.pkl'))

In [4]:
gd_monthly_df.keys()

Index(['gvkey', 'year', 'month', 'num_reviews', 'rating_mean',
       'career_opp_mean', 'comp_benefits_mean', 'management_mean',
       'work_life_mean', 'culture_values_mean', 'diversity_mean',
       'GD_CompanyName', 'GD_CompanyID', 'at', 'ceq', 'csho', 'dlc', 'dltt',
       'intan', 'pi', 'pifo', 'ppent', 'seq', 'spi', 'tlcf', 'txdfed', 'txdfo',
       'txdi', 'txpd', 'txr', 'txt', 'xrd', 'xsga', 'sic', 'state', 'ipodate',
       'conml'],
      dtype='object')

In [4]:
gd_ann_df = gd_monthly_df.groupby(['gvkey', 'year']).agg({
    'num_reviews': 'count', 'career_opp_mean': 'mean', 'comp_benefits_mean': 'mean', 'management_mean': 'mean',
    'rating_mean': 'mean', 'work_life_mean': 'mean', 'culture_values_mean': 'mean', 'diversity_mean': 'mean',
    'GD_CompanyName': 'first', 'GD_CompanyID': 'first'
    }).reset_index()

In [5]:
# Compute weighted annual averages for Glassdoor metrics
metrics = [
    'rating_mean',
    'career_opp_mean',
    'comp_benefits_mean',
    'management_mean',
    'work_life_mean',
    'culture_values_mean',
    'diversity_mean'
]

# Weighted totals per month
for col in metrics:
    gd_monthly_df[f'{col}_wt_total'] = gd_monthly_df[col] * gd_monthly_df['num_reviews']

# Aggregate to annual at gvkey-year level
agg_dict = {'num_reviews': 'sum', 'GD_CompanyName': 'first', 'GD_CompanyID': 'first'}
for col in metrics:
    agg_dict[f'{col}_wt_total'] = 'sum'

gd_ann_wt_df = gd_monthly_df.groupby(['gvkey', 'year']).agg(agg_dict).reset_index()

# Compute weighted averages; handle zero-review years
_den = gd_ann_wt_df['num_reviews'].replace(0, np.nan)
for col in metrics:
    gd_ann_wt_df[col] = gd_ann_wt_df[f'{col}_wt_total'] / _den
    gd_ann_wt_df.drop(columns=f'{col}_wt_total', inplace=True)

In [8]:
gd_ann_df.describe()

,gvkey,year,num_reviews,career_opp_mean,comp_benefits_mean,management_mean,rating_mean,work_life_mean,culture_values_mean,diversity_mean,GD_CompanyID
count,26748.000000,26748.000000,26748.000000,25903.000000,25653.000000,26031.000000,26748.000000,25823.000000,20666.000000,5302.000000,2.674800e+04
mean,59132.033760,2015.448744,7.117841,2.986580,3.285156,2.880023,3.239274,3.324475,3.203021,3.599401,1.772627e+05
std,64921.874347,4.117687,4.356761,0.763764,0.722401,0.824573,0.762293,0.767743,0.819674,0.713624,4.684169e+05
min,1004.000000,2008.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,4.000000e+00
25%,10894.000000,2012.000000,3.000000,2.571429,2.938571,2.432769,2.851668,2.950000,2.759259,3.276239,1.697000e+03
50%,26906.000000,2016.000000,8.000000,3.000000,3.322018,2.932102,3.272727,3.333333,3.241235,3.685693,6.135000e+03
75%,108797.000000,2019.000000,12.000000,3.446905,3.750000,3.333333,3.722222,3.812500,3.715504,4.000000,1.579100e+04
max,351038.000000,2023.000000,12.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,8.131118e+06


In [7]:
gd_ann_wt_df.describe()

,gvkey,year,num_reviews,GD_CompanyID,rating_mean,career_opp_mean,comp_benefits_mean,management_mean,work_life_mean,culture_values_mean,diversity_mean
count,26748.000000,26748.000000,26748.000000,2.674800e+04,26748.000000,26748.000000,26748.000000,26748.000000,26748.000000,26748.000000,26748.000000
mean,59132.033760,2015.448744,141.755533,1.772627e+05,3.246518,2.760671,2.967016,2.682577,3.035234,2.349330,0.561175
std,64921.874347,4.117687,813.901197,4.684169e+05,0.760414,0.941828,1.003587,0.946269,0.992414,1.489763,1.245529
min,1004.000000,2008.000000,1.000000,4.000000e+00,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,10894.000000,2012.000000,3.000000,1.697000e+03,2.857143,2.333333,2.550000,2.200000,2.625000,1.000000,0.000000
50%,26906.000000,2016.000000,13.000000,6.135000e+03,3.283333,2.916667,3.100000,2.778264,3.153958,2.823431,0.000000
75%,108797.000000,2019.000000,60.000000,1.579100e+04,3.732129,3.333333,3.600000,3.213787,3.636364,3.467317,0.000000
max,351038.000000,2023.000000,57350.000000,8.131118e+06,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000


In [6]:
gd_ann_wt_df.to_pickle(os.path.join(const.TEMP_PATH, 'glassdoor_ann_2008_2023.pkl'))

In [2]:
# Load major customer file
major_corp_cus_df = pd.read_stata(os.path.join(const.MAJ_DATA_PATH, 'MajorCorpCusMeaures_0624.dta'))
major_cus = pd.read_stata(os.path.join(const.MAJ_DATA_PATH, 'MajorCusMeaures_0624.dta'))

In [5]:
major_cus_df = major_corp_cus_df.merge(
    major_cus, on=['gvkey', 'fyear'], how='outer', suffixes=('_chen2025', '_cg2017')
)

In [9]:
major_cus_df.rename(columns={'fyear': 'year'}, inplace=True)

In [13]:
gd_ann_wt_df = pd.read_pickle(os.path.join(const.TEMP_PATH, 'glassdoor_ann_2008_2023.pkl'))
maj_with_gd_df = major_cus_df.merge(
    gd_ann_wt_df, on=['gvkey', 'year'], how='left'
)
gd_ann_wt_df[const.YEAR] -= 1
maj_with_gd_lag_df = maj_with_gd_df.merge(
    gd_ann_wt_df, on=['gvkey', 'year'], how='left', suffixes=('', '_l1')
)
maj_with_gd_lag_df.to_stata(os.path.join(
    const.MAJ_OUTPUT_PATH, 'MajorCustomer_Glassdoor_2008_2023_0110.dta'), write_index=False, version=119)

In [35]:
import wrds
# Connect to WRDS
import os
db = wrds.Connection(wrds_username='wangyouan')
print('WRDS connected:', db is not None)

Loading library list...
Done
WRDS connected: True


In [20]:
# Query Compustat annual fundamentals (North America)
comp_query = """
SELECT gvkey, fyear, datadate, at, lt, che, ceq, txditc, csho, prcc_f, mkvalt, sale, xrd, capx, ppent, ppegt, oibdp, ib, sich, dvpsx_f, dltt, dlc
FROM comp.funda
WHERE indfmt = 'INDL'
  AND datafmt = 'STD'
  AND consol  = 'C'
  AND popsrc  = 'D'
  AND fyear BETWEEN 2006 AND 2023
"""
funda = db.raw_sql(comp_query)
funda.rename(columns={'fyear': 'year'}, inplace=True)
print('Compustat rows:', len(funda))

# Basic cleaning
funda = funda.drop_duplicates(subset=['gvkey','year'], keep='last')
funda = funda.sort_values(['gvkey','year'])


Compustat rows: 206597


In [21]:
# Compute common firm control variables using lagged assets as denominators
import numpy as np
ctrl = funda.copy()

# Coerce numeric fields and handle missing values
for col in ['at','lt','che','ib','sale','xrd','capx','ppent','ppegt','csho','prcc_f','mkvalt','dvpsx_f','dltt','dlc']:
    ctrl[col] = pd.to_numeric(ctrl[col], errors='coerce')

# Replace missing R&D with 0
ctrl['xrd'] = ctrl['xrd'].fillna(0)

# Guard against non-positive assets
ctrl.loc[ctrl['at'] <= 0, 'at'] = np.nan

# Sort and compute lags (restrict to consecutive years)
ctrl = ctrl.sort_values(['gvkey','year']).copy()
ctrl['at_lag'] = ctrl.groupby('gvkey')['at'].shift(1)
ctrl['sale_lag'] = ctrl.groupby('gvkey')['sale'].shift(1)
ctrl['year_diff'] = ctrl.groupby('gvkey')['year'].diff()
ctrl.loc[ctrl['year_diff'] != 1, ['at_lag','sale_lag']] = np.nan
ctrl.loc[ctrl['at_lag'] <= 0, 'at_lag'] = np.nan

# Market capitalization
ctrl['mktcap'] = ctrl['prcc_f'] * ctrl['csho']

# Size: ln(total assets) [current]
ctrl['size_ln_at'] = np.log(ctrl['at'])

# Leverage: total liabilities / lagged assets
ctrl['leverage'] = ctrl['lt'] / ctrl['at_lag']

# Tobin's Q (approx): (MktCap + LT - CHE) / lagged assets
ctrl['tobinq_m'] = (ctrl['mktcap'] + ctrl['lt'] - ctrl['che']) / ctrl['at_lag']
ctrl['tobinq_alt'] = (ctrl['mkvalt'] + ctrl['lt'] - ctrl['che']) / ctrl['at_lag']
ctrl['tobinq'] = ctrl['tobinq_m'].fillna(ctrl['tobinq_alt'])

# Cash holdings: CHE / lagged assets
ctrl['cash_at'] = ctrl['che'] / ctrl['at_lag']

# ROA: IB / lagged assets
ctrl['roa'] = ctrl['ib'] / ctrl['at_lag']

# R&D intensity: XRD / lagged assets
ctrl['rd_at'] = ctrl['xrd'] / ctrl['at_lag']
ctrl['rd_indicator'] = (ctrl['xrd'] > 0).astype('float')

# CapEx intensity: CAPX / lagged assets
ctrl['capx_at'] = ctrl['capx'] / ctrl['at_lag']

# Asset tangibility: PPENT/lagged assets (fallback to PPEGT)
ctrl['tangibility'] = ctrl['ppent'] / ctrl['at_lag']
ctrl.loc[ctrl['tangibility'].isna(), 'tangibility'] = ctrl['ppegt'] / ctrl['at_lag']

# Debt to sales: (dltt + dlc) / SALE
ctrl['total_debt'] = ctrl['dltt'].fillna(0) + ctrl['dlc'].fillna(0)
ctrl['xtd_sale'] = ctrl['total_debt'] / ctrl['sale']

# Sales growth: (SALE - SALE_lag) / SALE_lag
ctrl['sale_growth'] = (ctrl['sale'] - ctrl['sale_lag']) / ctrl['sale_lag']

# Keep relevant columns
ctrl_vars = ctrl[['gvkey','year','datadate','sich','at','at_lag','lt','che','ib','sale','sale_lag','xrd','capx','ppent','ppegt','csho','prcc_f','mkvalt','mktcap','dvpsx_f','dltt','dlc',
                   'size_ln_at','leverage','tobinq','cash_at','roa','rd_at','rd_indicator','capx_at','tangibility','xtd_sale','sale_growth']]
print('Control vars shape:', ctrl_vars.shape)

Control vars shape: (206586, 33)


d:\anaconda3\envs\GlassDoor\Lib\site-packages\pandas\core\arrays\masked.py:672: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs2, **kwargs)


In [22]:
# Compute annual stock return from Compustat prcc_f incl. dividends (dvpsx_f)
ctrl_vars = ctrl_vars.sort_values(['gvkey','year']).copy()
ctrl_vars['prcc_f_lag'] = ctrl_vars.groupby('gvkey')['prcc_f'].shift(1)
ctrl_vars['year_diff'] = ctrl_vars.groupby('gvkey')['year'].diff()
ctrl_vars.loc[ctrl_vars['year_diff'] != 1, 'prcc_f_lag'] = np.nan
ctrl_vars['dvpsx_f'] = pd.to_numeric(ctrl_vars['dvpsx_f'], errors='coerce').fillna(0)
ctrl_vars['stock_return'] = (ctrl_vars['prcc_f'] + ctrl_vars['dvpsx_f']) / ctrl_vars['prcc_f_lag'] - 1
print('Computed stock_return rows:', ctrl_vars['stock_return'].notna().sum())

Computed stock_return rows: 164013


In [25]:
# Construct firm age: IPO year (ipodate) fallback to first Compustat year
ipo_query = """
SELECT gvkey, ipodate
FROM comp.company
"""
ipo_df = db.raw_sql(ipo_query)
ipo_df['ipodate'] = pd.to_datetime(ipo_df['ipodate'], errors='coerce')
ipo_df['ipo_year'] = ipo_df['ipodate'].dt.year

# First appearance in Compustat (earliest fiscal year)
first_comp = funda.groupby('gvkey', as_index=False)['year'].min().rename(columns={'year':'first_comp_year'})

# Determine first_year: IPO year if available, else earliest Compustat year
first_year_df = first_comp.merge(ipo_df[['gvkey','ipo_year']], on='gvkey', how='left')
first_year_df['first_year'] = first_year_df['ipo_year'].fillna(first_year_df['first_comp_year'])

# Merge into controls and compute firm_age
ctrl_vars = ctrl_vars.merge(first_year_df[['gvkey','first_year']], on='gvkey', how='left')
ctrl_vars['firm_age'] = ctrl_vars['year'] - ctrl_vars['first_year'] + 1
ctrl_vars.loc[ctrl_vars['firm_age'] < 1, 'firm_age'] = 1
print('Firm age computed; non-missing count:', ctrl_vars['firm_age'].notna().sum())

Firm age computed; non-missing count: 206586


In [26]:
# Save firm controls (winsorized 1%-99%, infinities -> NaN)
import numpy as np

# Select final columns (uses sich and Compustat-based stock_return and firm_age)
final_cols = ['gvkey','year','sich','firm_age',
              'size_ln_at','leverage','tobinq','cash_at','roa','rd_at','rd_indicator','capx_at','tangibility','xtd_sale','sale_growth','stock_return']
firm_controls = ctrl_vars[final_cols].copy()

# Replace infinities with NaN
firm_controls.replace([np.inf, -np.inf], np.nan, inplace=True)

# Winsorize numeric control variables at 1% and 99% (exclude firm_age)
num_cols = ['size_ln_at','leverage','tobinq','cash_at','roa','rd_at','capx_at','tangibility','xtd_sale','sale_growth','stock_return']
for col in num_cols:
    q_low = firm_controls[col].quantile(0.01)
    q_high = firm_controls[col].quantile(0.99)
    firm_controls[col] = firm_controls[col].clip(lower=q_low, upper=q_high)

# Persist to TEMP and optional Stata
temp_pkl = os.path.join(const.MAJ_TEMP_PATH, 'firm_controls_2008_2023.pkl')
firm_controls.to_pickle(temp_pkl)
print('Saved:', temp_pkl)

Saved: D:\Onedrive\Temp\Projects\MajorCustomer\temp\firm_controls_2008_2023.pkl


In [46]:
firm_controls.replace(pd.NA, np.nan, inplace=True)
base_reg_df = pd.read_stata(os.path.join(
    const.MAJ_OUTPUT_PATH, 'MajorCustomer_Glassdoor_2008_2023_0110.dta'))
firm_controls[const.GVKEY] = firm_controls['gvkey'].astype(int)
reg_with_ctrl_df = base_reg_df.merge(firm_controls, on=['gvkey', 'year'], how='left')
reg_with_ctrl_df['ln_num_reviews'] = np.log(reg_with_ctrl_df['num_reviews'] + 1)
reg_with_ctrl_df['ln_num_reviews_l1'] = np.log(reg_with_ctrl_df['num_reviews_l1'] + 1)
print('Final regression df shape:', reg_with_ctrl_df.shape)
reg_with_ctrl_df.to_stata(os.path.join(const.MAJ_OUTPUT_PATH, '20260111_maj_customer_gd_with_ctrl.dta'),
                          write_index=False, version=119)

Final regression df shape: (57908, 48)


In [47]:
reg_with_ctrl_df.shape

(57908, 48)

In [39]:
import wrds
# Connect to WRDS
import os
db = wrds.Connection(wrds_username='wangyouan')
print('WRDS connected:', db is not None)

Loading library list...
Done
WRDS connected: True


In [ ]:
# Append company names (conm, conml) to regression output
names_query = """
SELECT gvkey, conm, conml
FROM comp.company
"""
names_df = db.raw_sql(names_query)

In [ ]:
names_df['gvkey'] = pd.to_numeric(names_df['gvkey'], errors='coerce')
names_df = names_df.dropna(subset=['gvkey']).copy()
names_df['gvkey'] = names_df['gvkey'].astype(int)

# Merge into current regression DataFrame (reg_with_ctrl_df)
reg_with_names_df = reg_with_ctrl_df.merge(names_df[['gvkey','conm','conml']], on='gvkey', how='left')

# Save updated regression file (new dated filename)
out_with_names = os.path.join(const.MAJ_OUTPUT_PATH, '20260111_maj_customer_gd_with_ctrl.dta')
reg_with_names_df.to_stata(out_with_names, write_index=False, version=119)
print('Saved with names:', out_with_names)

Saved with names: D:\Onedrive\Temp\Projects\MajorCustomer\regression_data\20260111_maj_customer_gd_with_ctrl.dta


In [50]:
# Clean regression data: fill specified columns with 0 and drop missing ratings
fill_cols = ['cc_all_chen2025','cc_rank_chen2025','majorcus_dummy_chen2025','majorcus_sales_chen2025','maxcus_sales_chen2025',
             'cc_all_cg2017','cc_rank_cg2017','majorcus_dummy_cg2017','majorcus_sales_cg2017','maxcus_sales_cg2017']
if 'reg_with_ctrl_df' in globals():
    for c in [c for c in fill_cols if c in reg_with_ctrl_df.columns]:
        reg_with_ctrl_df[c] = pd.to_numeric(reg_with_ctrl_df[c], errors='coerce').fillna(0)
    # Drop rows missing either rating_mean or rating_mean_l1
    reg_with_ctrl_df = reg_with_ctrl_df.dropna(subset=['rating_mean','rating_mean_l1'], how='all')
    print('After cleaning, rows:', len(reg_with_ctrl_df))
else:
    raise RuntimeError('reg_with_ctrl_df not found; run the merge cell first.')

# Re-merge names (if available) and save final regression file
if 'names_df' in globals():
    reg_with_names_df = reg_with_ctrl_df.merge(names_df[['gvkey','conm','conml']], on='gvkey', how='left')
else:
    # Fallback: save without names
    reg_with_names_df = reg_with_ctrl_df.copy()

final_out = os.path.join(const.MAJ_OUTPUT_PATH, '20260111_maj_customer_gd_with_ctrl.dta')
reg_with_names_df.to_stata(final_out, write_index=False, version=119)
print('Saved cleaned regression file:', final_out)

After cleaning, rows: 17257
Saved cleaned regression file: D:\Onedrive\Temp\Projects\MajorCustomer\regression_data\20260111_maj_customer_gd_with_ctrl.dta


In [51]:
matched_df = reg_with_names_df[['GD_CompanyName', 'GD_CompanyID', 'conm', 'conml']].dropna(how='any').drop_duplicates()
matched_df.to_csv(os.path.join(const.MAJ_OUTPUT_PATH, '20260111_maj_customer_gd_name_matches.csv'), index=False)

In [53]:
gd_ann_wt_df = pd.read_pickle(os.path.join(const.TEMP_PATH, 'glassdoor_ann_2008_2023.pkl'))

In [55]:
drop_id_set = (10119, 932115, 6332, 321095, 149642, 5065, 466935, 10906, 443, 39601, 5016, 1370962, 12886, 2778962, 614260, 465735, 614365, 6046, 1105066, 1820653, 1513285, 7190, 200, 3601, 15700, 698007, 10093, 397779, 936577, 612889, 934625, 5960, 397407, 11940, 748887, 321095, 8285, 653845, 12929, 3054, 6589, 9930, 23719, 7532, 231829, 8448, 378578, 1369476, 319777, 378391,929490, 7057, 3230, 2849, 231790, 6730, 617613, 7603, 8992, 23704, 708, 464408, 719757, 2461, 195741, 934879, 5027, 10804, 9742, 465470, 760795, 1371340, 228277, 530322, 760684, 1368843, 529403, 3802, 526113, 322103, 935654, 267534, 602520, 466049, 8807, 10910, 661662, 10645, 9132, 12472, 884, 4523, 9726, 11063, 467314, 6446, 5864, 10089, 453116, 3154, 929846, 35325, 460850, 392802, 8466, 8619, 9641, 6412, 936786, 11011, 8731, 13076, 468058, 7639, 3798545, 1455628, 195435, 196804, 191183, 396837, 1513707, 282120, 3054, 6315, 933225, 305732, 1297571, 1236, 4944, 8223, 10369, 934705, 467838, 4725, 393458, 8131065, 1106881, 267538, 529117, 12304, 267604, 12810, 378631, 2565, 322103, 2248182, 9341, 11576, 9491, 1370689, 665867, 12287, 613686, 6721, 4193, 196838, 465455, 12586)
gd_ann_wt_df2 = gd_ann_wt_df[~gd_ann_wt_df['GD_CompanyID'].isin(drop_id_set)].copy()
gd_ann_wt_df2.shape

(25969, 12)

In [59]:
gd_ann_wt_df2.to_stata(os.path.join(const.MAJ_OUTPUT_PATH, 'glassdoor_ann_2008_2023.dta'),
                        write_index=False, version=119)